# Mini-Project 5 — Audit an MCP Server
### IT7075: Applied AI for Cybersecurity · Model Context Protocol

MCP is a contract. A server advertises tools; a client — or an LLM agent — discovers them and calls
them, trusting whatever comes back. So the interesting question is not *can you build a server*, it
is **what happens when the tool is quietly wrong**.

You have a working MCP server with a recon tool, a CVE-lookup tool, and a knowledge-base resource.
It has real defects. Your job is to find them by measurement, fix what code can fix, and say what
code cannot.

**MCP has two sides, so your five TODOs are split across two files — 12 lines in total:**

| | Where | What | Lines |
|---|---|---|---|
| **TODO 1** | `project_mcp_server.py` | a version-aware `lookup_cve_v2` tool | 2 |
| **TODO 2** | `project_mcp_server.py` | expose the KB as a **resource** | 1 |
| **TODO 3** | this notebook | connect over stdio and **discover** the tools | 3 |
| **TODO 4** | this notebook | **call** a tool | 3 |
| **TODO 5** | this notebook | **read** the resource | 3 |

Find them by searching for `###################` in **both** files.

> **Runs fully offline.** Simulated scans, a local knowledge base, no LLM — so no API key, no
> model, no GPU, and no packets on the network. The numbers you report are exact.

> **The server restarts every time.** The client launches `project_mcp_server.py` as a fresh
> subprocess on each connection, so edit the server file, re-run a client cell, and your change is
> live. Nothing to restart.

## Part 1 — Setup: running async MCP code inside Jupyter

MCP's client API is asynchronous, and Jupyter already owns an event loop. `run_async` — the helper
from the Lecture 3 notebook — runs a coroutine on a worker thread with its own loop. Given to you;
nothing to change.

In [ ]:
import sys, os, json, re, asyncio, threading
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


def run_async(coro):
    """GIVEN (Lecture 3): run a coroutine to completion on a worker thread,
    because Jupyter already owns the main event loop."""
    box = {}

    def worker():
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            box["value"] = loop.run_until_complete(coro)
        except Exception as e:                       # noqa: BLE001
            box["error"] = e
        finally:
            loop.close()

    t = threading.Thread(target=worker)
    t.start()
    t.join()
    if "error" in box:
        raise box["error"]
    return box["value"]


# ---- where things live ---------------------------------------------------
def project_path(name):
    """The project files sit next to this notebook, or one level up from solutions/."""
    return name if os.path.exists(name) else os.path.join("..", name)


SERVER_FILE = project_path("project_mcp_server.py")
EXPECTED_FILE = project_path("expected_cves.json")

# How the client starts the server: as a subprocess, over stdio.
server = StdioServerParameters(command=sys.executable, args=[SERVER_FILE])
ERRLOG = open("mcp_server_stderr.log", "w")     # a real file: Jupyter dislikes pipes

print("server file :", SERVER_FILE)
print("python      :", sys.executable)

## Part 2 — Connect and discover  ·  **TODO 3**

This is the MCP handshake, and it is the whole point of the protocol: the client does not know what
the server can do until it asks. Read the descriptions it prints — that text is the entire contract
an LLM would use to decide which tool to call.

> **Expected output:** four tools — `port_scan`, `list_targets`, `lookup_cve`, `lookup_cve_v2`.

In [ ]:
###################  TODO 3 — connect over stdio and list the tools  ###################
# WHAT: finish discover() so it returns [(tool name, first line of its docstring), ...]
#
# HOW (the Lecture 3 "Connect & Discover" cell) -- three lines inside the two
# `async with` blocks that are already open for you:
#
#   1. await session.initialize()                  the MCP handshake
#   2. tools = await session.list_tools()          ask what the server exposes
#   3. return [(t.name, t.description.splitlines()[0]) for t in tools.tools]
#
# Note `tools.tools`: list_tools() returns a result OBJECT with a .tools list on it.
################

async def discover():
    async with stdio_client(server, errlog=ERRLOG) as (read, write):
        async with ClientSession(read, write) as session:
            ____                                                                # <<< 1 line
            ____                                                                # <<< 1 line
            return ____                                                         # <<< 1 line


for name, description in run_async(discover()):
    print("- %-16s %s" % (name, description))

## Part 3 — Call a tool and read the resource  ·  **TODO 4 and TODO 5**

Two different MCP primitives. A **tool** is something the agent *does* (`call_tool`); a **resource**
is something it *reads* (`read_resource`). Same server, same session, different verbs.

> **Expected output:** a scan of `10.0.0.5` listing vsftpd, OpenSSH and Apache; then the first lines
> of the CVE knowledge base.

In [ ]:
###################  TODO 4 — call a tool  ###################
# WHAT: run the `port_scan` tool on `target` and keep its text in `scan`.
#
# HOW (the Lecture 3 "Call the Tools" cell):
#       result = await session.call_tool("port_scan", {"target": target})
#       scan   = result.content[0].text
#
# The second argument is a DICT of the tool's parameters, keyed by name -- exactly
# the names in the server's function signature. Getting one wrong is the most common
# MCP client bug, and the server will tell you so.
################

###################  TODO 5 — read a resource  ###################
# WHAT: fetch the knowledge base the server exposes at the URI "kb://cve" into `kb`.
#
# HOW:  kb = await session.read_resource("kb://cve")
#       ...and the text is then kb.contents[0].text (see the return line below).
#
# Note the difference from a tool call: read_resource takes a URI and no arguments,
# and the payload is under .contents (plural) rather than .content.
################

async def scan_and_read(target):
    async with stdio_client(server, errlog=ERRLOG) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = ____                                                       # <<< 1 line
            scan = ____                                                         # <<< 1 line
            kb = ____                                                           # <<< 1 line
            return scan, kb.contents[0].text


scan_text, kb_text = run_async(scan_and_read("10.0.0.5"))
print(scan_text)
print("\nRESOURCE kb://cve — first 180 characters:")
print(kb_text[:180], "...")

## Part 4 — Sweep the lab  *(no TODO — run it and read it)*

The coding is done; everything from here is measurement. Scan all four hosts, then ask **both**
lookup tools about every service that turned up.

> **Expected output:** 11 services across the four hosts.

In [ ]:
SERVICE_LINE = re.compile(r"^\s*(\d+)/tcp\s+open\s+(\S+)\s+(.*)$")


def parse_services(scan_text):
    """GIVEN: pull (port, service, version-banner) out of nmap-style output."""
    found = []
    for line in scan_text.splitlines():
        m = SERVICE_LINE.match(line)
        if m:
            found.append((m.group(1), m.group(2), m.group(3).strip()))
    return found


async def sweep(hosts, tool_names):
    """GIVEN: one session; scan every host, then ask each lookup tool about
    every service banner found."""
    out = []
    async with stdio_client(server, errlog=ERRLOG) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            for host in hosts:
                scan = (await session.call_tool("port_scan", {"target": host})).content[0].text
                for port, svc, banner in parse_services(scan):
                    row = {"host": host, "port": port, "banner": banner}
                    for tool in tool_names:
                        res = await session.call_tool(tool, {"service": banner})
                        row[tool] = res.content[0].text
                    out.append(row)
    return out


HOSTS = ["10.0.0.5", "10.0.0.8", "10.0.0.12", "10.0.0.20"]
rows = run_async(sweep(HOSTS, ["lookup_cve", "lookup_cve_v2"]))
print("discovered %d services across %d hosts\n" % (len(rows), len(HOSTS)))
for r in rows:
    print("  %-10s %-5s %s" % (r["host"], r["port"], r["banner"]))

## Part 5 — Score both tools  *(no TODO)*

`expected_cves.json` records, for every service in the lab, which knowledge-base section *should*
come back — or `null` where the KB genuinely has no entry. Comparing each tool's answer with that
sorts every response into one of four buckets:

| | Meaning | Why it matters |
|---|---|---|
| **grounded** | the right advisory came back | what you want |
| **blind spot** | "no known CVEs" — but the KB *does* cover it | the agent is reassured, wrongly |
| **wrong version** | an advisory for a *different* version | worse than silence: it looks authoritative |
| **true gap** | "no known CVEs" and the KB really has nothing | honest — fix it with knowledge, not code |

> **Expected output:**
>
> | tool | grounded | blind spot | wrong version | true gap |
> |---|---:|---:|---:|---:|
> | `lookup_cve` (the lecture's) | 6/9 | **2** | **1** | 2 |
> | `lookup_cve_v2` (yours) | **9/9** | 0 | 0 | 2 |
>
> (11 services in all: 9 the knowledge base can cover, plus 2 it genuinely cannot.)
>
> Look at *which* ones the lecture version got wrong. `Linux telnetd` and
> `Redis key-value store 4.0.9` were **already in the knowledge base** — it simply failed to match
> them, and told the agent there was nothing to worry about. And `Apache httpd 2.2.8` came back with
> the advisory for **2.4.49**, a version that host is not running.
>
> Note *how* that last one fails. The naive tool returns the right OpenSSL advisory **and** the
> 2.4.49 one together — so it is not simply wrong, it is right with a fabricated extra finding
> attached. An analyst chasing a path-traversal bug on a server that cannot have it has lost just as
> much time as one who missed something.
>
> Your version fixes all three, and the two remaining gaps (nginx, PostgreSQL) are honest: the
> knowledge base has no entry. **No amount of matching logic fixes those** — that is Part 6.

In [ ]:
GROUND_TRUTH = {s["service"]: s for s in json.load(open(EXPECTED_FILE))["services"]}


def classify(answer, service):
    """GIVEN: sort one tool answer into a bucket, given the ground truth.

    Note the `must_not` check: an answer can contain the right advisory AND an
    advisory for a version the host is not running. That still counts as a wrong
    version -- a report that names a vulnerability which is not there costs an
    analyst just as much time as one that misses a real one.
    """
    truth = GROUND_TRUTH.get(service, {})
    expected = truth.get("expected")
    said_nothing = answer.startswith("No known CVEs")
    if expected is None:
        return "true gap" if said_nothing else "wrong version"
    if said_nothing:
        return "blind spot"
    for forbidden in truth.get("must_not", []):
        if forbidden.lower() in answer.lower():
            return "wrong version"
    return "grounded" if expected.lower() in answer.lower() else "wrong version"


BUCKETS = ["grounded", "blind spot", "wrong version", "true gap"]
summary = {}
for tool in ["lookup_cve", "lookup_cve_v2"]:
    counts = dict.fromkeys(BUCKETS, 0)
    for r in rows:
        r[tool + "_verdict"] = classify(r[tool], r["banner"])
        counts[r[tool + "_verdict"]] += 1
    summary[tool] = counts

print("%-16s %-9s %-11s %-14s %s" % ("tool", "grounded", "blind spot", "wrong version", "true gap"))
for tool, c in summary.items():
    print("%-16s %-9d %-11d %-14d %d"
          % (tool, c["grounded"], c["blind spot"], c["wrong version"], c["true gap"]))

print("\nwhere they disagree:")
print("%-38s %-15s %s" % ("service", "lookup_cve", "lookup_cve_v2"))
for r in rows:
    if r["lookup_cve_verdict"] != r["lookup_cve_v2_verdict"]:
        print("%-38s %-15s %s" % (r["banner"][:36], r["lookup_cve_verdict"], r["lookup_cve_v2_verdict"]))

with open("results.csv", "w", encoding="utf-8") as f:
    f.write("host,port,service,lookup_cve,lookup_cve_v2\n")
    for r in rows:
        f.write("%s,%s,\"%s\",%s,%s\n" % (r["host"], r["port"], r["banner"],
                                          r["lookup_cve_verdict"], r["lookup_cve_v2_verdict"]))
print("\nwrote results.csv")

## Part 6 — Close the real gaps  *(edit `project_cve_kb.md`, then re-run)*

Two services come back honest-but-empty: **nginx 1.18.0** and **PostgreSQL 9.6.24**. Better matching
cannot help — the knowledge base has nothing to match. This is the other half of a tool: its data.

**Add two sections to `project_cve_kb.md`**, in the same shape as the ones already there:

```markdown
# nginx 1.18.0
One or two sentences: what the software is, a real CVE or CWE that affects this
version, its severity, the fix, and the port it usually listens on.
```

Then re-run Parts 4 and 5. `true gap` should fall to 0 and `grounded` rise to 11/11.

> **Now be careful, because this lever cuts both ways.** Write a *vague* entry — a bare `# nginx`
> with no version — and it will match every nginx banner ever, including versions that were never
> vulnerable. Try it deliberately: make one entry too generic, re-run, and watch a `wrong version`
> appear. Report what you saw. **A tool is its logic, its data, and its docstring — and all three
> can lie.**

In [ ]:
rows = run_async(sweep(HOSTS, ["lookup_cve", "lookup_cve_v2"]))

print("%-16s %-9s %-11s %-14s %s" % ("tool", "grounded", "blind spot", "wrong version", "true gap"))
for tool in ["lookup_cve", "lookup_cve_v2"]:
    counts = dict.fromkeys(BUCKETS, 0)
    for r in rows:
        counts[classify(r[tool], r["banner"])] += 1
    print("%-16s %-9d %-11d %-14d %d"
          % (tool, counts["grounded"], counts["blind spot"],
             counts["wrong version"], counts["true gap"]))

print("\n(If nothing changed, you have not saved project_cve_kb.md yet — the server")
print(" re-reads it on every call, so no restart is needed.)")

## Part 7 — Audit what the server exposes  *(no TODO)*

One last MCP-specific question, straight from the Lecture 3 safety section. An agent will call
anything a server advertises, so **the advertised surface is the attack surface**. Look at what this
server hands to any client that connects.

In [ ]:
async def audit():
    async with stdio_client(server, errlog=ERRLOG) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = (await session.list_tools()).tools
            resources = (await session.list_resources()).resources
            return tools, resources


tools, resources = run_async(audit())
print("This server advertises %d tools and %d resource(s):\n" % (len(tools), len(resources)))
for t in tools:
    params = list((t.inputSchema or {}).get("properties", {}))
    doc = (t.description or "").strip().splitlines()
    print("  %-16s params=%-22s doc lines=%d" % (t.name, ",".join(params) or "none", len(doc)))
    print("      %s" % (doc[0] if doc else "(NO DESCRIPTION -- an agent is guessing)"))
for r in resources:
    print("  resource %s  (%s)" % (r.uri, r.name))

print("\nFor your write-up, answer these from the output above:")
for q in ["which tools take free-text input from the model, and what would a bad value do?",
          "does any tool touch the network or the filesystem, and how would a client know?",
          "is any description vague enough that an agent could call the wrong tool?",
          "what would you refuse to expose over MCP at all?"]:
    print("  -", q)

## Part 8: your own server, your own tool, your own measurement

Everything above used the provided server and the provided lab, and it was practice. The rest of this
notebook is your own, and no code is provided for it.

First extend the server. In `project_mcp_server.py`:

1. Add one host of your own to `_SIMULATED_HOSTS`, with three realistic service banners, product and
   version in nmap style. Choose them so at least one is covered by your knowledge base and at least
   one is not.
2. Add a tool of your own that an analyst would find useful, for example one that reports the
   severity of a finding, checks a port against a policy list, or summarizes what a host exposes.
   Write its docstring as carefully as its code: the docstring is what a client and a model read.
3. Add the missing knowledge to `project_cve_kb.md` for the services that return nothing, with real
   products and real CVEs, cited in `SOURCES.md`, and record the expected section for each of your
   services in `expected_cves.json`.

Then write the client cells below to:

4. Connect to your server, discover the tools, and show that your new tool is advertised.
5. Sweep your own host with both lookup tools and with your new tool, and sort every answer into the
   four buckets: grounded, blind spot, wrong version, and true gap. Save the table as
   `results_mine.csv`.
6. Make one knowledge base entry deliberately vague by dropping the version, run the sweep again, and
   record what changed.

> When you are finished, run the whole notebook top to bottom and save it with the outputs showing,
> then commit it, the server, the knowledge base, and your results to your private GitHub repository.
> See `MiniProject_MCP.md` section 6.1 for the folder structure.

In [ ]:
###################  YOUR SERVER: CONNECT AND DISCOVER  ###################
# Connect to your extended server, list the tools it advertises, and show
# your new tool and its description.
################

### Your measurement

In [ ]:
###################  YOUR MEASUREMENT  ###################
# Sweep your own host, sort every answer into grounded, blind spot, wrong
# version, or true gap, and save the table as results_mine.csv. Then make one
# knowledge base entry vague, run it again, and record what changed.
################

---

## What to write up

One PDF report and a video of five to ten minutes. See `MiniProject_MCP.md` for the full
requirements. The questions that matter:

- The blind spot is the dangerous bucket. A tool that says no known CVEs about a service the
  knowledge base covers is worse than an error message. Why, and what would a client have to do to
  notice?
- A wrong version answer looks authoritative. Which of the two failures would you rather ship, and
  what does your answer say about how much an agent should trust a tool result?
- Two levers. What did better matching fix, what needed new knowledge, and what broke when you made
  an entry vague on purpose? Give the numbers before and after.
- The docstring is the contract. Which description on your server would you rewrite before letting a
  model use it, and which tool would you refuse to expose at all?
- A tool over MCP compared with a tool inside your agent. In the LangGraph module a tool was a Python
  function you called yourself. Here it is a separate process a client discovers. What did you gain,
  what did it cost, and when would you keep the function?

Submit one PDF report with the link to your project folder and the link to your video on its first
page. The notebook, the server, the knowledge base, and your results live in the repository, which is
private, with your instructor and the TA added as collaborators. The notebook must be committed with
its outputs showing.